#### **Problem Statement**

- Retrieval Augmented Generation is a prompting technique for optimizing the output of a large language model, by - referencing a knowledge source in addition to the training data sources before a response is generated.

- Why is Retrieval Augmented Generation important? : Improves **factual consistency** in response and making the response more trustworthy, addressing the problem of hallucination in generative AI.

- RAG Assessment Tool (RAGAS) is a framework for testing and evaluating QA Pipeline.

- RAG assessment consists of metrics for generation and retrieval.

- Generation - category is divided into faithfulness (is the response grounded in the retrieved context, measures hallucination / factual accuracy) and answer relevance (does the response address the question)

- Retrieval - is divided into Context Precision (signal to noise ratio of retrieved context) and Context Recall (can it retrieve all the relevant information required to provide a response to the question)

- The harmonic mean of these four aspects is the RAGAS score

-  Langsmith complements Ragas by being a supportive platform for visualizing and for delving into the
evaluation results for RAGAS metrics to be explainable and reproducible.

#### **References :**

[1] https://www.promptingguide.ai/techniques/rag

[2] Lewis, Patrick, et al. "Retrieval-augmented generation for knowledge-intensive nlp tasks." Advances in Neural Information Processing Systems 33 (2020): 9459-9474.

[3] https://aws.amazon.com/what-is/retrieval-augmented-generation/

[4] https://www.kaggle.com/datasets/radek1/sci-or-not-sci-hypthesis-testing-pack

[5] https://arxiv.org/pdf/2309.15217.pdf

[6] https://arxiv.org/pdf/2401.12379.pdf

[7] https://docs.ragas.io/en/latest/howtos/integrations/langchain.html

[8] https://blog.langchain.dev/evaluating-rag-pipelines-with-ragas-langsmith/

#### **Installing Depedencies**

In [ ]:
# Installing dependencies : Langchain framework, Faiss vector database, tiktoken, datasets, pypdf, ragas, sentence transformers

# Pinned versions (tested Oct 2026) so this notebook keeps working on Google Colab
!pip install -q langchain==0.1.20 langchain-community==0.0.38 langchain-core==0.1.52 langchain-text-splitters==0.0.2 langchain-openai==0.1.7 openai==1.58.1 tiktoken==0.14.0 ragas==0.0.22 datasets==5.0.1 langsmith==0.1.147 numpy==1.26.4 unstructured==0.27.10 pypdf==6.19.0 faiss-cpu==1.15.1 sentence-transformers
!sed -i "s/gpt-3.5-turbo-16k/gpt-4o-mini/g" $(python -c "import importlib.util,os;print(os.path.dirname(importlib.util.find_spec('ragas').origin))")/llms/*.py  # ragas 0.0.22 defaults to a retired OpenAI model; point it to gpt-4o-mini
# !pip install langchain -q   # (replaced by the pinned install above)
# !pip install langchain-openai -q   # (replaced by the pinned install above)
# !pip install tiktoken -q   # (replaced by the pinned install above)
# !pip install langchain_community -q   # (replaced by the pinned install above)
# !pip install unstructured -q   # (replaced by the pinned install above)
# !pip install pypdf -q   # (replaced by the pinned install above)
# !pip install faiss-cpu -q   # (replaced by the pinned install above)
# !pip install sentence-transformers -q   # (replaced by the pinned install above)
# !pip install datasets -q   # (replaced by the pinned install above)
# !pip install ragas==0.0.22 -q   # (replaced by the pinned install above)
# !pip install transformers -q   # (replaced by the pinned install above)
# !pip install torch -q   # (replaced by the pinned install above)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.9/815.9 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 43.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 kB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This beh

#### **Importing required libraries**

In [ ]:
from langchain_community.document_loaders import WebBaseLoader #WebBaseLoader loads text from HTML webpages in a document format
from langchain_community.embeddings import HuggingFaceEmbeddings # HuggingFace Embeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter #Text Splitter
from langchain_community.vectorstores import FAISS #FAISS vector database
from langchain_openai import ChatOpenAI #OpenAI Interface from Langchain
from langchain_core.runnables import RunnablePassthrough  #Allows to pass an input unchanged
from langchain_core.output_parsers import StrOutputParser #Parses llm response to be in a specific format
from langchain_core.prompts import ChatPromptTemplate # Langchain prompt template for chat models
from langchain.chains import RetrievalQA # RetrievalQA chain

#### **API KEY**

In [ ]:
# Extracting OPENAI_API_KEY using userdata.get()

from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [ ]:
# Set the environment variable : Langchain API Key and OpenAI API Key
from uuid import uuid4
import os
unique_id = uuid4().hex[0:8]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"RAGAS_Evaluation"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = "ls__xxxxxxxxxxxxxxxxxxxxxxxxxx"
os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxx"

#### **Document Loader in Langchain**


- Document loaders : Load the data from a specific source in the form of a document.
(a document consists of text with associated metadata)
- Example: TextLoader, ArxivLoader, PyPDFLoader, WebBaseLoader and more.


- https://python.langchain.com/docs/modules/data_connection/document_loaders/

In [ ]:
# WebBaseLoader extracts text from HTML webpages in a document format
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
docs = loader.load()

#### **Text Splitter**

- Recursive character text split : splits text by recursively looking at characters.

- https://api.python.langchain.com/en/latest/text_splitter/langchain.text_splitter.RecursiveCharacterTextSplitter.html

In [ ]:
# Recursive Character text splitter, with chunk size : 500 and chunk overlap of 20
# # creating meaningful chunks of text with some overlap for ensuring context between the chunks.


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 20,
    length_function = len,
    is_separator_regex = False
)

text_split = text_splitter.split_documents(docs)

#### **Embeddings**

- The data is broken into meaningful chunks. These need to be converted into a vector representation, which involves transforming the text data into embeddings which are numeric representations that capture the semantic meaning behind text.

- Examples : OpenAI Embeddings, HuggingFaceEmbeddings with specific models such as BERT.

- https://www.datacamp.com/blog/what-is-retrieval-augmented-generation-rag

- https://python.langchain.com/docs/integrations/text_embedding/openai

- https://huggingface.co/sentence-transformers/all-mpnet-base-v2

In [ ]:
# Initialize HuggingFaceEmbeddings() with sentence-transformers text embeddings model
# HuggingFace API Key from within colab environment

embeddings_model = HuggingFaceEmbeddings(model_name= "sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### **Vector Store**

- FAISS is a library for efficient similarity search and clustering of dense vectors

- https://python.langchain.com/docs/integrations/vectorstores/faiss


In [ ]:
# FAISS.from_documents() method takes in text chunks with the appropriate embeddings model
db = FAISS.from_documents(text_split, embeddings_model)

#### **Retriever Interface**

- VectorStoreRetriever, backed by a vectorstore.
- When a query enters a system, it must also be converted into embeddings.
- The same model must be used for document and query embedding to ensure uniformity between the two.
- Two search types to retrieve text : Retrieve text based on similarity : uses similarity search in the retriever object, where it selects the text chunk vectors that are most similar (using cosine similarity or Euclidean distance) to the question vector.
- "MMR", maximum marginal relevance search, optimizing for similarity to query AND diversity among the selected documents.

- https://towardsdatascience.com/4-ways-of-question-answering-in-langchain-188c6707cc5


In [ ]:
# Vector store retriever
retriever = db.as_retriever(search_type="similarity", search_kwargs= {"k": 5})

#### **Prompt for retrieval augmented generation**

In [ ]:
# Providing a prompt to the large language model to guide the model response with context

prompt_template = """
                  Human : Answer honestly based on the context provided.
                  Otherwise, please say you don't know. Please do not falsify.

                  ### CONTEXT : {context}


                  ### QUESTION : {question}


                  \n


                  AI :

                  """

In [ ]:
# ChatOpenAI() method with the specified model : gpt-4o-mini
# ChatPromptTemplate() method with the prompt template to guide the large language model

model = ChatOpenAI(openai_api_key = OPENAI_API_KEY, model_name = "gpt-4o-mini")
prompt = ChatPromptTemplate.from_template(prompt_template)

In [ ]:
# Retrieval_Chain for QA with the retriever interface provided as context and the specified large language model
# Question passed unchanged using RunnablePassThrough() when the retrieval chain is invoked

retrieval_chain = (
    {
        "context" : retriever,
        "question" : RunnablePassthrough()
        }
    | prompt
    | model
    |StrOutputParser()
)
response = retrieval_chain.invoke("What specific model does Markov decision process have?")

In [ ]:
# model response
response

'The Markov decision process has a transition model that describes the probability that a particular action will change the state in a particular way, and a reward function that supplies the utility of each state and the cost of each action.'

#### **Evaluation using RAGAS**

- **Answer Relevancy** : How relevant is the generated answer to the question.
- **Faithfulness** : Factual accuracy of the generated answer.
- **Context Recall** : Quantifies if all of the relevant information required to answer the question was retrieved.

In [ ]:
# Importing Dataset from datasets library

from datasets import Dataset

In [ ]:
# question and ground truth pairs for RAG Evaluation

# Generating a list of questions and responses for evaluation
questions =[
    "What is Artificial Intelligence?",
    "What are some applications of Artificial Intelligence?",
    "Who is the first person to conduct substantial reseach in Machine Intelligence?",
    "What is knowledge base?",
    "What is an agent?",
    "What model does Markov Decision Process have?"
]

ground_truths=[
    ["Artificial intelligence is the intelligence of machines or software. It is a field of study in computer science that develops and studies intelligent machines"],
    ["Applications of Artificial Intelligence are web search engines (e.g. Google Search), recommendation systems (used by YouTube, Amazon and Netflix), interacting via human speech (such as Google Assistant, Siri and Alexa), self-driving cars (e.g. Waymo), generative and creative tools (ChatGPT, AI art), superhuman play and analysis in strategy games (such as Chess)"],
    ["Alan Turing"],
    ["A knowledge base is a body of knowledge in a form that can be used by a program"],
    ["An agent is anything that perceives and takes actions in the world"],
    ["A Markov Decision Process has a transition model that describes the probability that a particular action will change the state in a particular way"]
]


In [ ]:
# Inference : Preparation of remaining information, extracting answers from the model with the context.
answers = []
contexts = []

for query in questions:
  answers.append(retrieval_chain.invoke(query))
  contexts.append([docs.page_content for docs in retriever.get_relevant_documents(query)])

In [ ]:
# Creating a dictionary consisting of questions, extracted answers, context and groud truth

data = {
    "question":questions,
    "answer" : answers,
    "contexts": contexts,
    "ground_truths":ground_truths

}

dataset = Dataset.from_dict(data)

In [ ]:
# Importing evaluate and the relevant metrics from ragas library
from ragas import evaluate
from ragas.metrics import(
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
)

In [ ]:
# Quantifying faithfulness, context_recall, answer_relevancy
result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        context_recall,
        answer_relevancy
    ]
)

evaluating with [faithfulness]


100%|██████████| 1/1 [01:18<00:00, 78.10s/it]


evaluating with [context_recall]


100%|██████████| 1/1 [00:24<00:00, 24.40s/it]


evaluating with [answer_relevancy]


100%|██████████| 1/1 [00:05<00:00,  5.42s/it]


In [ ]:
result.to_pandas() # Quantified faithfulness, context_recall and answer_relevancy shown in a Pandas Dataframe

,question,answer,contexts,ground_truths,faithfulness,context_recall,answer_relevancy
0,What is Artificial Intelligence?,Artificial Intelligence is the intelligence of...,[Glossary\nGlossary\nvte\nArtificial intellige...,[Artificial intelligence is the intelligence o...,1.0,1.0,0.993445
1,What are some applications of Artificial Intel...,Some applications of Artificial Intelligence i...,[AI technology is widely used throughout indus...,[Applications of Artificial Intelligence are w...,1.0,1.0,0.999999
2,Who is the first person to conduct substantial...,Alan Turing was the first person to conduct su...,[Alan Turing was the first person to conduct s...,[Alan Turing],1.0,1.0,0.991422
3,What is knowledge base?,A knowledge base is a body of knowledge repres...,[A knowledge base is a body of knowledge repre...,[A knowledge base is a body of knowledge in a ...,1.0,1.0,0.993704
4,What is an agent?,An agent is anything that perceives and takes ...,"[An ""agent"" is anything that perceives and tak...",[An agent is anything that perceives and takes...,1.0,1.0,0.960690
5,What model does Markov Decision Process have?,The Markov Decision Process has a transition m...,[A Markov decision process has a transition mo...,[A Markov Decision Process has a transition mo...,1.0,1.0,0.898674


#### **Evaluation Using Langsmith**

- With Langsmith, the evaluation results as part of the RagasEvaluatorChain can be monitored.

In [ ]:
# Importing RagasEvaluatorChain
from ragas.langchain.evalchain import RagasEvaluatorChain
from langsmith import Client
from langsmith.utils import LangSmithError
from langchain.smith import RunEvalConfig, run_on_dataset

In [ ]:
# Dataset for Langsmith Evaluation
# question and ground truth pairs for RAG Evaluation :  Generating a list of questions and responses for evaluation
questions =[
    "What is Artificial Intelligence?",
    "What are some applications of Artificial Intelligence?",
    "Who is the first person to conduct substantial reseach in Machine Intelligence?",
    "What is knowledge base?",
    "What is an agent?",
    "What model does Markov Decision Process have?"
]

ground_truths=[
    "Artificial intelligence is the intelligence of machines or software. It is a field of study in computer science that develops and studies intelligent machines",
    "Applications of Artificial Intelligence are web search engines (e.g. Google Search), recommendation systems (used by YouTube, Amazon and Netflix), interacting via human speech (such as Google Assistant, Siri and Alexa), self-driving cars (e.g. Waymo), generative and creative tools (ChatGPT, AI art), superhuman play and analysis in strategy games (such as Chess)",
    "Alan Turing",
    "A knowledge base is a body of knowledge in a form that can be used by a program",
    "An agent is anything that perceives and takes actions in the world",
    "A Markov Decision Process has a transition model that describes the probability that a particular action will change the state in a particular way"
]


In [ ]:
# creating question and ground truth pair
examples = [
    {"query": q, "ground_truths": [ground_truths[i]]}
    for i, q in enumerate(questions)
]
examples

[{'query': 'What is Artificial Intelligence?',
  'ground_truths': ['Artificial intelligence is the intelligence of machines or software. It is a field of study in computer science that develops and studies intelligent machines']},
 {'query': 'What are some applications of Artificial Intelligence?',
  'ground_truths': ['Applications of Artificial Intelligence are web search engines (e.g. Google Search), recommendation systems (used by YouTube, Amazon and Netflix), interacting via human speech (such as Google Assistant, Siri and Alexa), self-driving cars (e.g. Waymo), generative and creative tools (ChatGPT, AI art), superhuman play and analysis in strategy games (such as Chess)']},
 {'query': 'Who is the first person to conduct substantial reseach in Machine Intelligence?',
  'ground_truths': ['Alan Turing']},
 {'query': 'What is knowledge base?',
  'ground_truths': ['A knowledge base is a body of knowledge in a form that can be used by a program']},
 {'query': 'What is an agent?',
  'gr

#### **Creating Evaluator Chain**

In [ ]:
# Evaluation chain for RAGAS assessment :
faithfulness_chain = RagasEvaluatorChain(metric=faithfulness) # Faithfulness metric
answer_relevancy_chain = RagasEvaluatorChain(metric=answer_relevancy) # Answer Relevancy
context_rel_chain = RagasEvaluatorChain(metric=context_precision) # context precision

In [ ]:
# Dataset for Langsmith Evaluation
# https://blog.langchain.dev/evaluating-rag-pipelines-with-ragas-langsmith/

client = Client() # Initializing LangSmith Client()
dataset_name = "Artificial Intelligence Wikipedia Page"

try:
  # Check if dataset exists
  dataset = client.read_dataset(dataset_name = dataset_name)
  print("Using existing dataset : ", dataset.name)

except LangSmithError:
  # if not create a new one, with the generated query example
  dataset = client.create_dataset(
      dataset_name = dataset_name,
      description = "Artificial Intelligence Wikipedia Page"
  )
  for e in examples:
    client.create_example(
        inputs = {"query": e["query"]},
        outputs = {"ground_truths": e["ground_truths"]},
        dataset_id = dataset.id,
    )
  print("Created a new dataset: ", dataset.name)

Using existing dataset :  Artificial Intelligence Wikipedia Page


In [ ]:
# factory function create_qa_chain() :

def create_qa_chain(return_context=True):
  qa_chain = RetrievalQA.from_chain_type(
      ChatOpenAI(temperature=0.2),
      retriever=retriever,
      return_source_documents=return_context,
  )
  return qa_chain

In [ ]:
# RunEvalConfig() method, with the custom evaluators specified :

evaluation_config = RunEvalConfig(
    custom_evaluators = [
        faithfulness_chain,
        answer_relevancy_chain,
        context_rel_chain,
    ],
    prediction_key = "result"
)

# run_on_dataset() method
result = run_on_dataset(
    client,
    dataset_name,
    create_qa_chain,
    evaluation=evaluation_config,
    input_mapper=lambda x:x
)

/usr/local/lib/python3.10/dist-packages/langchain/smith/evaluation/runner_utils.py:1284: LangChainPendingDeprecationWarning: The input_mapper argument is deprecated and will be removed in a future release. Please add a  RunnableLambda to your chain to map inputs to the expected format instead. Example:
def construct_chain():
    my_chain = ...
    input_mapper = {'other_key': 'MyOtherInput', 'my_input_key': x}
    return input_mapper | my_chain
run_on_dataset(..., llm_or_chain_factory=construct_chain)
(See https://api.python.langchain.com/en/latest/schema/langchain.schema.runnable.base.RunnableLambda.html)
  warn_deprecated("0.0.305", message=_INPUT_MAPPER_DEP_WARNING, pending=True)


View the evaluation results for project 'left-station-57' at:
https://smith.langchain.com/o/44f4dfa2-b9e4-582c-a0c7-ddbf06b019aa/datasets/6f814f43-7ef8-4ee0-bb7e-cbc9cd0d46da/compare?selectedSessions=0af65099-766a-430d-abe6-c1d2ce6a4814

View all tests for Dataset Artificial Intelligence Wikipedia Page at:
https://smith.langchain.com/o/44f4dfa2-b9e4-582c-a0c7-ddbf06b019aa/datasets/6f814f43-7ef8-4ee0-bb7e-cbc9cd0d46da
[>                                                 ] 0/6

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `__call__` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


[------------------------------------------------->] 6/6